# RQ2 dynamic-allocation pilot — seed 4, T4 x2

Runs three seed-4 jobs: Geometry-Dynamic p=1 with 50+50, matched-compute Resource-Dynamic with 50+50, and an additional Geometry-Dynamic control trained continuously for 100 epochs with one optimizer/scheduler. T4 x2 runs two jobs at once and queues the third. Only `test-rq2` must be attached; both CPU policies are regenerated from seeds 0/1/2 before seed 4 is touched. Test data remains sealed.

In [ ]:
import os, subprocess, sys, time, json, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
PILOT_SEED = 4
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop>=0.1.1'], check=True)
import torch
assert torch.cuda.device_count() == 2, f'Choose GPU T4 x2; detected {torch.cuda.device_count()}'
print('Commit:', subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip())
print([torch.cuda.get_device_name(i) for i in range(2)])

## Rebuild and freeze the two policies before seed 4 training

In [ ]:
import importlib, rq2_anchor_placement, rq2_theory_allocation_probe, rq2_probabilistic_support, rq2_dynamic_seed3_pilot
rq2_anchor_placement = importlib.reload(rq2_anchor_placement)
rq2_theory_allocation_probe = importlib.reload(rq2_theory_allocation_probe)
rq2_probabilistic_support = importlib.reload(rq2_probabilistic_support)
rq2_dynamic_seed3_pilot = importlib.reload(rq2_dynamic_seed3_pilot)
RQ2_INPUT = Path('/kaggle/input/notebooks/dyhngg/test-rq2')
assert RQ2_INPUT.exists(), f'Attach RQ2-v1 output: {RQ2_INPUT}'
RQ2_ROOT = rq2_anchor_placement.find_rq2_development_root(RQ2_INPUT, '/kaggle/working/materialized-rq2-dynamic-seed4')
RUN_DIR = Path('/kaggle/working/rq2-dynamic-seed4-pilot')
THEORY_ROOT = RUN_DIR/'policy_derivation'/'theory-allocation-probe'
PREVIEW_ROOT = RUN_DIR/'policy_derivation'/'probabilistic-support-preview'
theory_summary = rq2_theory_allocation_probe.run_theory_probe(RQ2_ROOT, THEORY_ROOT, waiting_steps=2_000_000)
preview_summary = rq2_probabilistic_support.build_support_policy(RQ2_ROOT, PREVIEW_ROOT, pi_min=1e-8)
assert theory_summary['training_authorized'] is False and preview_summary['training_authorized'] is False
assert all(value for key, value in theory_summary.items() if key.endswith('_pass'))
assert all(preview_summary['assertions'].values())
print('Policy derivation passed; geometry total compute / Uniform:', preview_summary['geometry_total_compute_ratio_vs_uniform'])

## Train seed 4: two 50+50 methods plus one continuous-100 geometry control

In [ ]:
import scripts.run_dynamic_seed3_pilot as dynamic_runner
dynamic_runner = importlib.reload(dynamic_runner)
started = time.perf_counter()
result = dynamic_runner.run_seed_pilot(RQ2_ROOT, THEORY_ROOT, PREVIEW_ROOT, RUN_DIR, gpu_ids=[0,1], seed=PILOT_SEED, include_geometry_continuous=True)
print(f'Seed-4 pilot completed in {(time.perf_counter()-started)/3600:.2f} hours')
print(json.dumps(result['decision'], indent=2))

In [ ]:
import pandas as pd
from IPython.display import display
display(pd.read_csv(RUN_DIR/'pretraining_sampler_sanity.csv'))
display(pd.read_csv(RUN_DIR/'dynamic_pilot_method_summary.csv'))
display(pd.read_csv(RUN_DIR/'dynamic_pilot_width_comparison.csv'))
for method in ['geometry_dynamic','resource_dynamic','geometry_dynamic_continuous']:
    history = pd.read_csv(RUN_DIR/method/f'seed_{PILOT_SEED}'/'training_epoch_metrics.csv')
    assert set(history.epoch.astype(int)) == set(range(1,101))
    print(method, 'schedule:', history.schedule.unique().tolist(), 'phase rows:', history.groupby('phase').size().to_dict())
    print(method, 'final cumulative realized/expected:', history.iloc[-1].cumulative_realized_total_flops_per_batch/history.iloc[-1].expected_total_flops_per_batch)

## Export checkpoints, logs, frozen policies, and validation results

In [ ]:
required = [RUN_DIR/'dynamic_pilot_decision.json', RUN_DIR/'dense_validation_accuracy.csv', RUN_DIR/'dynamic_pilot_method_summary.csv', RUN_DIR/'dynamic_pilot_width_comparison.csv']
for method in ['geometry_dynamic','resource_dynamic','geometry_dynamic_continuous']:
    required += [RUN_DIR/method/f'seed_{PILOT_SEED}'/name for name in ['epoch_050.pt','epoch_100.pt','latest.pt','training_epoch_metrics.csv','width_inclusion_by_epoch.csv','pair_counts_by_epoch.csv','training_provenance.json']]
missing = [str(path) for path in required if not path.is_file() or path.stat().st_size == 0]
assert not missing, f'Missing seed-4 artifacts: {missing}'
bundle_path = Path('/kaggle/working/rq2-dynamic-seed4-pilot.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as bundle:
    for path in RUN_DIR.rglob('*'):
        if path.is_file() and path.name != 'checkpoint.pt':
            bundle.write(path, path.relative_to(RUN_DIR))
print('Download/persist:', bundle_path, f'{bundle_path.stat().st_size/2**30:.2f} GiB')
bundle_path